## Example 1: SecretsManagerClient Type Annotations for Secret Management

In [ ]:
from types_aiobotocore_secretsmanager import SecretsManagerClient
from types_aiobotocore_secretsmanager.type_defs import ListSecretsResponseTypeDef, SecretListEntryTypeDef, DescribeSecretResponseTypeDef
from aiobotocore.session import get_session
from botocore.exceptions import ClientError, NoCredentialsError

async def demonstrate_secretsmanager_client_types():
    session = get_session()
    async with session.create_client("secretsmanager", region_name="us-east-1") as client:
        client: SecretsManagerClient  # Type annotation enables IDE autocomplete
        try:
            secrets_response: ListSecretsResponseTypeDef = await client.list_secrets(MaxResults=10)
            print(f"Secrets Manager Secrets: {len(secrets_response.get('SecretList', []))} secrets")
            for secret in secrets_response.get("SecretList", [])[:3]:
                secret_typed: SecretListEntryTypeDef = secret
                print(f"  Secret: {secret_typed.get('Name')} (ARN: {secret_typed.get('ARN', 'N/A')[:50]}...)")
            if secrets_response.get('SecretList'):
                secret_name = secrets_response['SecretList'][0]['Name']
                describe_response: DescribeSecretResponseTypeDef = await client.describe_secret(SecretId=secret_name)
                print(f"\nSecret described: {describe_response.get('Name')} (last changed: {describe_response.get('LastChangedDate')})")
        except (NoCredentialsError, ClientError) as e:
            print(f"No credentials or access denied (expected): {type(e).__name__}")
        print("SecretsManagerClient provides 30+ typed methods for secret management")

await demonstrate_secretsmanager_client_types()

## Example 2: Using Paginator Types for Secrets Manager Resources

In [ ]:
from types_aiobotocore_secretsmanager import SecretsManagerClient
from types_aiobotocore_secretsmanager.paginator import ListSecretsPaginator
from aiobotocore.session import get_session
from botocore.exceptions import ClientError, NoCredentialsError

async def demonstrate_secretsmanager_paginator_types():
    session = get_session()
    async with session.create_client("secretsmanager", region_name="us-east-1") as client:
        client: SecretsManagerClient
        secrets_paginator: ListSecretsPaginator = client.get_paginator("list_secrets")
        print("Secrets Manager Paginator types:")
        print(f"  ListSecretsPaginator: {type(secrets_paginator).__name__}")
        try:
            secret_count = 0
            async for page in secrets_paginator.paginate(PaginationConfig={'MaxItems': 50}):
                secret_count += len(page.get("SecretList", []))
            print(f"Found {secret_count} secrets using paginator (first 50 at most)")
        except (NoCredentialsError, ClientError) as e:
            print(f"No credentials or access denied (expected): {type(e).__name__}")
        print("Package provides ListSecretsPaginator for handling large secret lists")

await demonstrate_secretsmanager_paginator_types()

## Example 3: Using TypeDefs for Secret Creation and Storage

In [ ]:
from types_aiobotocore_secretsmanager.type_defs import CreateSecretRequestTypeDef, TagTypeDef, CreateSecretResponseTypeDef, PutSecretValueRequestTypeDef
from types_aiobotocore_secretsmanager.type_defs import GetSecretValueResponseTypeDef, ListSecretsResponseTypeDef, SecretListEntryTypeDef
from types_aiobotocore_secretsmanager import SecretsManagerClient
from types_aiobotocore_secretsmanager.paginator import ListSecretsPaginator
import json

def demonstrate_secret_creation_typedefs():
    tags: list[TagTypeDef] = [{"Key": "Environment", "Value": "Production"}, {"Key": "Application", "Value": "WebApp"}]
    secret_data = {"username": "admin", "password": "super-secret-password", "host": "db.example.com", "port": 5432}
    create_secret: CreateSecretRequestTypeDef = {
        "Name": "prod/database/credentials",
        "Description": "Production database credentials",
        "SecretString": json.dumps(secret_data),
        "KmsKeyId": "arn:aws:kms:us-east-1:123456789012:key/abcd1234-ab12-cd34-ef56-123456789012",
        "Tags": tags
    }
    print("Type-safe Secret Creation Request:")
    print(f"  Secret Name: {create_secret['Name']}")
    print(f"  Description: {create_secret['Description']}")
    print(f"  KMS Key: {create_secret['KmsKeyId'][-40:]}")
    print(f"  Tags: {len(tags)} tags")
    for tag in tags:
        print(f"    {tag['Key']}={tag['Value']}")
    response: CreateSecretResponseTypeDef = {"ARN": "arn:aws:secretsmanager:us-east-1:123456789012:secret:prod/database/credentials-AbCdEf", "Name": create_secret['Name'], "VersionId": "EXAMPLE1-90ab-cdef-fedc-ba987EXAMPLE"}
    print(f"\nSecret Created: ARN={response['ARN'][-50:]}")
    print(f"Version ID: {response['VersionId']}")
    print("TypeDefs ensure type safety for Secrets Manager secret creation")

demonstrate_secret_creation_typedefs()

## Example 4: Using TypeDefs for Secret Rotation Configuration

In [ ]:
from types_aiobotocore_secretsmanager.type_defs import RotateSecretRequestTypeDef, RotationRulesTypeTypeDef, RotateSecretResponseTypeDef

def demonstrate_secret_rotation_typedefs():
    rotation_rules: RotationRulesTypeTypeDef = {
        "AutomaticallyAfterDays": 30,
        "Duration": "3h",
        "ScheduleExpression": "rate(30 days)"
    }
    rotate_secret: RotateSecretRequestTypeDef = {
        "SecretId": "prod/database/credentials",
        "RotationLambdaARN": "arn:aws:lambda:us-east-1:123456789012:function:SecretsManagerRotation",
        "RotationRules": rotation_rules,
        "RotateImmediately": False
    }
    print("Type-safe Secret Rotation Configuration:")
    print(f"  Secret ID: {rotate_secret['SecretId']}")
    print(f"  Lambda ARN: {rotate_secret['RotationLambdaARN'][-50:]}")
    print(f"  Rotation Frequency: Every {rotation_rules['AutomaticallyAfterDays']} days")
    print(f"  Rotation Duration: {rotation_rules['Duration']}")
    print(f"  Schedule Expression: {rotation_rules['ScheduleExpression']}")
    print(f"  Rotate Immediately: {rotate_secret['RotateImmediately']}")
    response: RotateSecretResponseTypeDef = {"ARN": "arn:aws:secretsmanager:us-east-1:123456789012:secret:prod/database/credentials-AbCdEf", "Name": rotate_secret['SecretId'], "VersionId": "EXAMPLE2-90ab-cdef-fedc-ba987EXAMPLE"}
    print(f"\nRotation Configured: Version ID={response['VersionId']}")
    print("TypeDefs ensure type safety for Secrets Manager rotation")

demonstrate_secret_rotation_typedefs()

## Example 5: Using TypeDefs for Secret Version Management

In [ ]:
from types_aiobotocore_secretsmanager.type_defs import UpdateSecretVersionStageRequestTypeDef, DescribeSecretResponseTypeDef, SecretVersionsListEntryTypeDef
import json

def demonstrate_secret_versioning_typedefs():
    update_version: UpdateSecretVersionStageRequestTypeDef = {
        "SecretId": "prod/database/credentials",
        "VersionStage": "AWSCURRENT",
        "MoveToVersionId": "EXAMPLE2-90ab-cdef-fedc-ba987EXAMPLE",
        "RemoveFromVersionId": "EXAMPLE1-90ab-cdef-fedc-ba987EXAMPLE"
    }
    print("Type-safe Secret Version Stage Update:")
    print(f"  Secret ID: {update_version['SecretId']}")
    print(f"  Version Stage: {update_version['VersionStage']}")
    print(f"  Move To Version: {update_version['MoveToVersionId']}")
    print(f"  Remove From Version: {update_version['RemoveFromVersionId']}")
    version_entry: SecretVersionsListEntryTypeDef = {
        "VersionId": "EXAMPLE2-90ab-cdef-fedc-ba987EXAMPLE",
        "VersionStages": ["AWSCURRENT"],
        "CreatedDate": "2024-01-15T10:00:00Z"
    }
    print(f"\nVersion Entry:")
    print(f"  Version ID: {version_entry['VersionId']}")
    print(f"  Stages: {', '.join(version_entry['VersionStages'])}")
    print(f"  Created: {version_entry['CreatedDate']}")
    print("TypeDefs ensure type safety for Secrets Manager version management")

demonstrate_secret_versioning_typedefs()